# Phase 2 — Temporal Graph Construction

**Objectif**: Transformer les données brutes CSV en un graphe temporel d'interactions compatible avec PyTorch Geometric et TGB.

Ce notebook documente:
1. Les concepts clés de la représentation temporelle
2. Le pipeline de prétraitement
3. La vérification de compatibilité avec TGB
4. La stratégie de sous-échantillonnage pour le développement

---
## 1. Concepts fondamentaux

### Graphe Temporel d'Interactions vs Graphe Statique

Un graphe statique classique **G = (V, E)** représente des relations permanentes. Pour les vols aériens, cela serait inapproprié car:
- Un vol Paris→NYC le lundi n'implique pas un vol permanent
- La fréquence des vols varie dans le temps (COVID, saisons)
- L'ordre chronologique contient de l'information prédictive

Notre représentation est un **Temporal Interaction Graph**:

$$G(t) = (V, E(t))$$

où chaque interaction est un tuple **(src, dst, t, feat)**:
- `src`: aéroport de départ (ID entier)
- `dst`: aéroport d'arrivée (ID entier)
- `t`: timestamp UNIX de l'interaction
- `feat`: attribut de l'arête (type d'avion encodé)

### Représentation event-based vs snapshots

Deux approches existent pour les graphes temporels:

| Approche | Description | Avantage | Inconvénient |
|----------|-------------|----------|--------------|
| **Snapshots** | Découper le temps en fenêtres (ex: 1 jour) | Simple | Perte de précision temporelle |
| **Event-based** | Chaque interaction est un événement horodaté | Précision maximale | Plus complexe |

Nous utilisons l'approche **event-based**, qui est celle attendue par TGN (Temporal Graph Network) et TGB.

### Graphe dirigé

Le graphe est **dirigé**: un vol LFPG→KJFK (Paris→NYC) ≠ KJFK→LFPG. C'est confirmé par l'analyse Phase 1 où la corrélation in-degree/out-degree n'est pas parfaite.

### Interactions répétées vs routes persistantes

Une distinction importante:
- **Interaction**: un vol spécifique à un instant t (ce que nos données représentent)
- **Route commerciale**: une liaison régulière entre deux aéroports (agrégation de nos données)

La même paire (src, dst) peut apparaître des milliers de fois avec des timestamps différents. TGN exploite cette répétition pour apprendre les patterns temporels.

---
## 2. Pipeline de prétraitement

Le module `src/data/preprocessing.py` implémente un pipeline en 4 étapes:

1. **Node mapping**: Conversion des codes ICAO en IDs entiers contigus (ordre de première apparition dans le CSV, identique à TGB)
2. **Typecode mapping**: Encodage des types d'avion (0=manquant, 1=rare, 2+=connu)
3. **Edge conversion**: Lecture chunked du CSV 2.1 GB → tableaux numpy
4. **Temporal split**: Découpage train/val/test par quantiles de timestamps

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timezone

sns.set_theme(style='whitegrid', palette='muted')

from src.graph.temporal_graph import TemporalFlightGraph

g = TemporalFlightGraph()
print(g)

---
## 3. Node Mapping: ICAO codes to integer IDs

### Pourquoi un mapping?

Les GNNs travaillent avec des indices entiers pour indexer les embeddings. Les codes ICAO (ex: KJFK, LFPG) doivent être convertis en entiers contigus [0, N-1].

### Convention TGB: first-seen order

TGB assigne les IDs dans **l'ordre de première apparition** en lisant le CSV chronologiquement. Nous reproduisons exactement cette convention pour être compatibles avec les negative samples pré-générés de TGB.

In [ ]:
mapping = g.node_mapping
print(f"Total airports mapped: {len(mapping):,}")
print(f"\nFirst 10 mappings (first airports to appear in chronological data):")
print(mapping.head(10).to_string(index=False))

print(f"\nSample lookups:")
for code in ['KJFK', 'LFPG', 'EGLL', 'KLAX', 'RJTT']:
    nid = g.node_id(code)
    print(f"  {code} -> node ID {nid}")

---
## 4. Node Features

Chaque noeud (aéroport) a un vecteur de 16 features:

| Feature | Dimensions | Encodage |
|---------|-----------|----------|
| Airport type | 7 | One-hot (balloonport, closed, heliport, large_airport, medium_airport, seaplane_base, small_airport) |
| Continent | 7 | One-hot (6 continents + missing) |
| Longitude | 1 | Normalisé [-1, 1] |
| Latitude | 1 | Normalisé [-1, 1] |

In [ ]:
nf = g.node_features
print(f"Node feature matrix shape: {nf.shape}")
print(f"  (num_nodes={nf.shape[0]:,}, num_features={nf.shape[1]})")
print(f"\nFeature statistics:")
print(f"  Min: {nf.min():.3f}")
print(f"  Max: {nf.max():.3f}")
print(f"  Mean: {nf.mean():.3f}")
print(f"  Non-zero: {(nf != 0).sum():,} / {nf.size:,} ({100*(nf != 0).mean():.1f}%)")

# Show features for JFK
jfk_id = g.node_id('KJFK')
print(f"\nKJFK (node {jfk_id}) features: {nf[jfk_id]}")

---
## 5. Edge Features: Aircraft Type Encoding

### Gestion des valeurs manquantes (40.7%)

Le typecode est manquant pour 40.7% des vols. Trois options:

1. **Supprimer les lignes**: perdre 40% des données = inacceptable
2. **Imputer**: risque de biais
3. **Encoder explicitement le manquant**: **notre choix**

Notre encodage:
- `0` = typecode manquant
- `1` = typecode rare (hors top 200)
- `2-201` = top 200 typecodes les plus fréquents

C'est un feature catégoriel unique (pas one-hot) car il sert d'attribut d'arête dans TGN, pas de feature discriminant principal.

In [ ]:
ef = np.load('data/processed/edge_feat.npy', mmap_mode='r')

# Distribution of encoded values
n_total = len(ef)
n_missing = (ef == 0).sum()
n_rare = (ef == 1).sum()
n_known = (ef >= 2).sum()

print(f"Edge feature encoding distribution:")
print(f"  Missing (0):   {n_missing:>12,} ({100*n_missing/n_total:.1f}%)")
print(f"  Rare    (1):   {n_rare:>12,} ({100*n_rare/n_total:.1f}%)")
print(f"  Known (2-201): {n_known:>12,} ({100*n_known/n_total:.1f}%)")
print(f"  Total:         {n_total:>12,}")

# Top typecodes
tc_mapping = pd.read_csv('data/processed/typecode_mapping.csv')
print(f"\nTop 10 aircraft types:")
print(tc_mapping.head(10).to_string(index=False))

---
## 6. Temporal Split: Train/Val/Test

### Chronological Split (anti-fuite d'information)

Le split est **strictement chronologique**: le modèle ne voit JAMAIS d'interactions futures pendant l'entraînement. C'est fondamental pour la prédiction temporelle.

TGB utilise des **quantiles de timestamps** (pas des proportions d'arêtes):
- `val_time = quantile(timestamps, 0.70)` 
- `test_time = quantile(timestamps, 0.85)`
- Train: t <= val_time
- Val: val_time < t <= test_time
- Test: t > test_time

Cela donne un split ~70/15/15 en timestamps, mais pas exactement en nombre d'arêtes (car la densité d'interactions varie dans le temps, notamment le creux COVID).

In [ ]:
stats = g.statistics()

print("Temporal Split Summary")
print("=" * 50)
print(f"Total edges: {stats['num_edges']:,}")
print(f"Time range:  {stats['time_start'][:10]} to {stats['time_end'][:10]}")
print()
print(f"Train: {stats['train_edges']:>12,} edges ({100*stats['train_edges']/stats['num_edges']:.1f}%)")
print(f"Val:   {stats['val_edges']:>12,} edges ({100*stats['val_edges']/stats['num_edges']:.1f}%)")
print(f"Test:  {stats['test_edges']:>12,} edges ({100*stats['test_edges']/stats['num_edges']:.1f}%)")

# Show split boundaries
timestamps = g.timestamps
val_time, test_time = np.quantile(np.array(timestamps), [0.70, 0.85])
print(f"\nSplit boundaries:")
print(f"  Val start:  {datetime.fromtimestamp(val_time, tz=timezone.utc).strftime('%Y-%m-%d')}")
print(f"  Test start: {datetime.fromtimestamp(test_time, tz=timezone.utc).strftime('%Y-%m-%d')}")

In [ ]:
# Visualize the split on the temporal distribution
from collections import Counter

daily_train = Counter()
daily_val = Counter()
daily_test = Counter()

ts_arr = np.array(timestamps)
train_mask = np.array(g.train_mask)
val_mask = np.array(g.val_mask)
test_mask = np.array(g.test_mask)

for label, mask, counter in [
    ('train', train_mask, daily_train),
    ('val', val_mask, daily_val),
    ('test', test_mask, daily_test),
]:
    days = (ts_arr[mask] // 86400).astype(np.int32)
    unique_days, counts = np.unique(days, return_counts=True)
    for d, c in zip(unique_days, counts):
        counter[int(d)] = int(c)

from datetime import timedelta
epoch = datetime(1970, 1, 1, tzinfo=timezone.utc)

fig, ax = plt.subplots(figsize=(14, 5))
for label, counter, color in [
    ('Train (70%)', daily_train, '#2196F3'),
    ('Validation (15%)', daily_val, '#FF9800'),
    ('Test (15%)', daily_test, '#F44336'),
]:
    days_sorted = sorted(counter.keys())
    dates = [epoch + timedelta(days=d) for d in days_sorted]
    vals = [counter[d] for d in days_sorted]
    ax.fill_between(dates, vals, alpha=0.5, color=color, label=label)
    ax.plot(dates, vals, color=color, linewidth=0.5, alpha=0.7)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Daily Interactions', fontsize=12)
ax.set_title('Temporal Train/Val/Test Split', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
os.makedirs('../results/figures', exist_ok=True)
plt.savefig('../results/figures/09_temporal_split.png', dpi=150)
plt.show()
print('Saved: results/figures/09_temporal_split.png')

---
## 7. TGB Compatibility Verification

Les negative samples pré-générés par TGB sont essentiels pour une évaluation comparable aux résultats publiés. Nous vérifions que nos IDs sont compatibles.

In [ ]:
import pickle, itertools

with open('../datasets/tgbl-flight/tgbl-flight_val_ns_v2.pkl', 'rb') as f:
    val_ns = pickle.load(f)

print(f"TGB val negative samples: {len(val_ns):,} entries")
print(f"Each entry: (src, dst, timestamp) -> 20 negative destinations")

# Verify compatibility: check 500 val edges
sources = g.sources
destinations = g.destinations
val_indices = np.where(np.array(g.val_mask))[0]

matches = 0
for idx in val_indices[:500]:
    s, d, t = int(sources[idx]), int(destinations[idx]), int(timestamps[idx])
    key = (np.int64(s), np.int64(d), np.int64(t))
    if key in val_ns:
        matches += 1

print(f"\nCompatibility check: {matches}/500 val edges found in TGB neg samples")
print("Status: COMPATIBLE" if matches == 500 else "Status: MISMATCH - investigate!")

del val_ns  # free ~1 GB RAM

---
## 8. PyTorch Geometric TemporalData

Le module `src/graph/temporal_graph.py` fournit un wrapper `TemporalFlightGraph` qui produit des objets `TemporalData` de PyG:

```
TemporalData(
    src: Tensor[N],     # source node IDs
    dst: Tensor[N],     # destination node IDs  
    t:   Tensor[N],     # timestamps
    msg: Tensor[N, 1],  # edge features (aircraft type)
)
```

C'est le format attendu par `TemporalDataLoader` et par TGN.

In [ ]:
# Demonstrate TemporalData creation for a small month
jan_2019 = g.get_monthly_subsample(2019, 1)
print(f"January 2019 subsample:")
print(f"  Edges: {jan_2019.src.shape[0]:,}")
print(f"  src dtype: {jan_2019.src.dtype}")
print(f"  t dtype: {jan_2019.t.dtype}")
print(f"  msg shape: {jan_2019.msg.shape}")

# Show it works with TemporalDataLoader
from torch_geometric.loader import TemporalDataLoader

loader = TemporalDataLoader(jan_2019, batch_size=200)
batch = next(iter(loader))
print(f"\nFirst batch from TemporalDataLoader:")
print(f"  batch.src: {batch.src.shape}")
print(f"  batch.dst: {batch.dst.shape}")
print(f"  batch.t:   {batch.t.shape}")
print(f"  batch.msg: {batch.msg.shape}")

---
## 9. Sub-sampling Strategy

Avec 67M d'interactions et ~2.8 GB de RAM libre, charger tout en mémoire pour l'entraînement est risqué. Stratégie:

**Pour le développement**: travailler sur un sous-ensemble temporel (ex: 1 mois ~ 1-2M interactions)  
**Pour l'évaluation finale**: utiliser les données complètes avec `TemporalDataLoader` qui charge les batches séquentiellement

La méthode `get_monthly_subsample(year, month)` permet de sélectionner rapidement une fenêtre temporelle.

In [ ]:
# Show monthly edge counts
from datetime import timedelta

monthly_counts = []
ts_arr = np.array(g.timestamps)

for year in range(2019, 2023):
    for month in range(1, 13):
        start = int(datetime(year, month, 1, tzinfo=timezone.utc).timestamp())
        if month == 12:
            end = int(datetime(year + 1, 1, 1, tzinfo=timezone.utc).timestamp()) - 1
        else:
            end = int(datetime(year, month + 1, 1, tzinfo=timezone.utc).timestamp()) - 1
        
        count = int(((ts_arr >= start) & (ts_arr <= end)).sum())
        if count > 0:
            monthly_counts.append((f"{year}-{month:02d}", count))

print("Monthly interaction counts:")
print(f"{'Month':>8}  {'Interactions':>12}")
print("-" * 24)
for m, c in monthly_counts:
    bar = '#' * max(1, c // 100000)
    print(f"{m:>8}  {c:>12,}  {bar}")

---
## 10. Summary

### Fichiers produits

| Fichier | Contenu | Taille |
|---------|---------|--------|
| `data/processed/sources.npy` | Source node IDs (int32) | 256 MB |
| `data/processed/destinations.npy` | Destination node IDs (int32) | 256 MB |
| `data/processed/timestamps.npy` | UNIX timestamps (int64) | 512 MB |
| `data/processed/edge_feat.npy` | Encoded typecodes (int32) | 256 MB |
| `data/processed/node_features.npy` | Node features (float32, 18143x16) | 1.1 MB |
| `data/processed/train_mask.npy` | Boolean train mask | 64 MB |
| `data/processed/val_mask.npy` | Boolean validation mask | 64 MB |
| `data/processed/test_mask.npy` | Boolean test mask | 64 MB |
| `data/processed/node_mapping.csv` | ICAO code <-> integer ID | 0.2 MB |
| `data/processed/typecode_mapping.csv` | Typecode <-> integer ID | <0.1 MB |

### Modules

- `src/data/preprocessing.py`: Pipeline CSV -> numpy (run once)
- `src/graph/temporal_graph.py`: `TemporalFlightGraph` wrapper

### Choix techniques justifiés

1. **Event-based** (pas snapshots): compatible TGN, précision maximale
2. **First-seen ID mapping**: compatible avec les negative samples TGB
3. **Timestamp quantile split**: reproduit exactement le split TGB
4. **Memory-mapped numpy**: charge les données sans tout mettre en RAM
5. **Missing typecode = catégorie explicite**: pas de perte de données
6. **Chunked reading**: contourne la limite RAM de 2.8 GB

In [ ]:
# Final summary table
print("Phase 2 - Temporal Graph Construction: COMPLETE")
print("=" * 55)
s = g.statistics()
print(f"Nodes (airports):     {s['num_nodes']:>12,}")
print(f"Edges (interactions): {s['num_edges']:>12,}")
print(f"Node features:        {s['num_node_features']:>12}")
print(f"Time range:           {s['time_start'][:10]} to {s['time_end'][:10]}")
print(f"Train edges:          {s['train_edges']:>12,}")
print(f"Val edges:            {s['val_edges']:>12,}")
print(f"Test edges:           {s['test_edges']:>12,}")
print(f"TGB compatible:       {'YES':>12}")